# Day 24: Model Explainability and Feature Importance

YBI Foundation — Python AI & Data Science Internship

**Project:** AI-Based Student Performance Prediction System

## Objective and theory

Explain the saved Day 17 decision tree with raw-column permutation importance and aggregated impurity importance. Permutation importance measures the change in Macro F1 after shuffling one column; impurity importance describes training splits. Neither establishes causality.

## Prerequisites

Install requirements-dev.txt in the active environment. The repository contains the restored model, evaluation dataset and source modules. The next cell locates the checkout or clones it for Colab. No new model is selected here.


In [1]:
from pathlib import Path
import os, sys, subprocess, json
root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if not (root / "src").exists():
    target = root / "IYB_INTERNSHIP"
    if not target.exists():
        subprocess.run(["git", "clone", "https://github.com/lightcode01-oss/IYB_INTERNSHIP.git", str(target)], check=True)
    root = target
os.chdir(root)
sys.path.insert(0, str(root))
print("Project:", root.name)


Project: IYB_INTERNSHIP


---
### 2. Environment Setup and Project Path Resolution

We configure the environment to support both local workspace execution and Google Colab environments.

In [2]:
import os
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve Project Root (works locally and in Colab)
PROJECT_ROOT = Path(".").resolve()
if (PROJECT_ROOT / "artifacts").exists():
    pass
elif (PROJECT_ROOT.parent / "artifacts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

pipeline_path = PROJECT_ROOT / "artifacts" / "student_performance_pipeline.joblib"
metadata_path = PROJECT_ROOT / "artifacts" / "student_performance_metadata.json"
data_path = PROJECT_ROOT / "day_16_error_analysis.csv"

print("=" * 70)
print("DAY 24: ENVIRONMENT & ARTIFACT DISCOVERY")
print("=" * 70)
print(f"Project Root Resolved: OK")
print(f"Pipeline Artifact Found: {pipeline_path.exists()} ({pipeline_path.name})")
print(f"Metadata Artifact Found: {metadata_path.exists()} ({metadata_path.name})")
print(f"Evaluation Dataset Found: {data_path.exists()} ({data_path.name})")

from src.explainability import (
    load_explainability_pipeline,
    load_reference_dataset,
    validate_feature_schema,
    compute_permutation_importance,
    compute_native_feature_importance,
    explain_individual_prediction,
    plot_permutation_importance,
    plot_native_importance,
    CANONICAL_FEATURES,
    FORBIDDEN_LEAKAGE_COLUMNS
)
print("Explainability Module: OK (src.explainability)")

DAY 24: ENVIRONMENT & ARTIFACT DISCOVERY
Project Root Resolved: OK
Pipeline Artifact Found: True (student_performance_pipeline.joblib)
Metadata Artifact Found: True (student_performance_metadata.json)
Evaluation Dataset Found: True (day_16_error_analysis.csv)


Explainability Module: OK (src.explainability)


---
### 3. Loading Model Pipeline and Verifying Preprocessing Architecture

We inspect the saved model artifact to confirm whether it contains an isolated estimator or an end-to-end scikit-learn `Pipeline` encapsulating preprocessing.

In [3]:
pipeline = load_explainability_pipeline(pipeline_path)
with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Loaded Object Type: {type(pipeline)}")
print(f"Pipeline Steps: {list(pipeline.named_steps.keys())}")

preprocessor = pipeline.named_steps["preprocessor"]
classifier = pipeline.named_steps["classifier"]

print(f"Preprocessor Step: {preprocessor.__class__.__name__}")
for name, trans, cols in preprocessor.transformers_:
    print(f"  - {name} ({trans.__class__.__name__}): {len(cols)} features")

print(f"Classifier Step: {classifier.__class__.__name__}")
print(f"  - Hyperparameters: {classifier.get_params(deep=False)}")
print(f"  - Target Classes: {list(classifier.classes_)}")
print(f"Total Transformed Features at Classifier Input: {len(preprocessor.get_feature_names_out())}")

Loaded Object Type: <class 'sklearn.pipeline.Pipeline'>
Pipeline Steps: ['preprocessor', 'classifier']
Preprocessor Step: ColumnTransformer
  - num_ord (FunctionTransformer): 13 features
  - cat (OneHotEncoder): 17 features
Classifier Step: DecisionTreeClassifier
  - Hyperparameters: {'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 10, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 10, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'random_state': 42, 'splitter': 'best'}
  - Target Classes: ['High', 'Low', 'Medium']
Total Transformed Features at Classifier Input: 56


---
### 4. Reference Evaluation Dataset & Leakage Safeguard Verification

To generate authentic explainability results without synthetic data or model re-fitting, we load the authentic held-out test split ($N=130$) established in Day 16.

**Leakage Safeguards Enforced:**
- Final examination mark `G3` is strictly excluded from feature matrix $X$.
- The derived target `performance_category` is strictly separated from inputs.
- Only the 30 canonical socio-demographic and academic attributes are provided to the pipeline.

In [4]:
X_test, y_test = load_reference_dataset(data_path)

print("Reference Dataset Inspection:")
print(f"  - Input Feature Matrix (X) Shape: {X_test.shape}")
print(f"  - Target Series (y) Shape: {y_test.shape}")

print("Class Distribution in Held-Out Test Set:")
print(y_test.value_counts())

print("\nLeakage Checks:")
for col in FORBIDDEN_LEAKAGE_COLUMNS:
    print(f"  - Forbidden column '{col}' in X: {col in X_test.columns} [PASSED]")

validate_feature_schema(X_test)
print("Schema Validation Status: PASSED (30/30 features verified)")

Reference Dataset Inspection:
  - Input Feature Matrix (X) Shape: (130, 30)
  - Target Series (y) Shape: (130,)
Class Distribution in Held-Out Test Set:
actual_category
Medium    84
High      26
Low       20
Name: count, dtype: int64

Leakage Checks:
  - Forbidden column 'G3' in X: False [PASSED]
  - Forbidden column 'performance_category' in X: False [PASSED]
Schema Validation Status: PASSED (30/30 features verified)


---
### 5. Permutation Feature Importance on Held-Out Test Data

#### Methodology
Permutation feature importance (`sklearn.inspection.permutation_importance`) assesses the dependency of the model on each input feature by measuring the degradation in the evaluation metric (**Macro F1**) when the values of that feature are randomly permuted across the test set.

#### Why Permute on Raw Input Columns (Pre-Pipeline)?
Because our model is wrapped in an end-to-end `Pipeline`, we permute raw columns before one-hot encoding. This avoids creating synthetic, invalid categorical states (e.g. permuting one-hot dummy flags independently could result in a student having multiple mother's jobs or none at all).

In [5]:
perm_df = compute_permutation_importance(
    pipeline=pipeline,
    X=X_test,
    y=y_test,
    scoring="f1_macro",
    n_repeats=15,
    random_state=42,
    n_jobs=1
)

print("=" * 70)
print("TOP 10 FEATURES BY PERMUTATION IMPORTANCE (HELD-OUT TEST SET)")
print("=" * 70)
print(f"{'Rank':<5} {'Feature':<15} {'Mean Macro F1 Drop':<20} {'Std Dev'}")
print("-" * 70)
for _, row in perm_df.head(10).iterrows():
    print(f"{int(row['rank']):<5} {row['feature']:<15} {row['importance_mean']:+.4f}{'':<13} ±{row['importance_std']:.4f}")
print("=" * 70)

[2026-11-02 13:42:02,569] INFO in model_explainability: Computing permutation importance with 15 repeats (scoring='f1_macro', seed=42)...


TOP 10 FEATURES BY PERMUTATION IMPORTANCE (HELD-OUT TEST SET)
Rank  Feature         Mean Macro F1 Drop   Std Dev
----------------------------------------------------------------------
1     failures        +0.0629              ±0.0186
2     Mjob            +0.0437              ±0.0094
3     freetime        +0.0286              ±0.0241
4     romantic        +0.0225              ±0.0124
5     famrel          +0.0166              ±0.0119
6     Fjob            +0.0148              ±0.0066
7     absences        +0.0144              ±0.0123
8     Pstatus         +0.0112              ±0.0044
9     Medu            +0.0107              ±0.0151
10    sex             +0.0107              ±0.0054


---
### 6. Native Feature Importance (Mean Decrease in Impurity)

The fitted `DecisionTreeClassifier` calculates feature importance based on Gini impurity reduction (MDI) across internal tree splits. Here we extract importances across all 56 transformed columns and aggregate them back to the 30 canonical feature names.

In [6]:
df_transformed, df_native_agg = compute_native_feature_importance(pipeline)

print("=" * 70)
print("DECISION TREE NATIVE FEATURE IMPORTANCE (MDI / GINI IMPURITY)")
print("=" * 70)
print(f"Total Features Analyzed: {len(df_native_agg)} aggregated, {len(df_transformed)} transformed")
print(f"Sum of Native Feature Importances: {df_native_agg['native_importance'].sum():.4f}\n")

print(f"{'Rank':<5} {'Feature':<15} {'Native Importance (Share of Impurity Reduction)'}")
print("-" * 70)
for _, row in df_native_agg.head(10).iterrows():
    print(f"{int(row['rank']):<5} {row['feature']:<15} {row['native_importance']:.4f}")
print("=" * 70)

DECISION TREE NATIVE FEATURE IMPORTANCE (MDI / GINI IMPURITY)
Total Features Analyzed: 30 aggregated, 56 transformed
Sum of Native Feature Importances: 1.0000

Rank  Feature         Native Importance (Share of Impurity Reduction)
----------------------------------------------------------------------
1     health          0.1029
2     failures        0.0893
3     studytime       0.0798
4     freetime        0.0685
5     reason          0.0683
6     absences        0.0551
7     age             0.0537
8     Mjob            0.0522
9     Walc            0.0472
10    famrel          0.0434


---
### 7. Visualizing Global Feature Importance

We visualize both **Permutation Feature Importance** and **Native Decision Tree Importance**.

In [7]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Permutation Importance Plot
top_perm = perm_df.head(10).iloc[::-1]
y_pos = np.arange(len(top_perm))
bars1 = ax1.barh(y_pos, top_perm["importance_mean"], xerr=top_perm["importance_std"], color="#2563EB", alpha=0.85, capsize=4)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(top_perm["feature"], fontsize=10)
ax1.set_xlabel("Mean Macro F1 Decrease (±1 Std Dev)", fontsize=10, fontweight="bold")
ax1.set_title("Permutation Importance (Held-Out Test Set)", fontsize=11, fontweight="bold")
ax1.axvline(0, color="#64748B", linestyle="--")
ax1.grid(axis="x", linestyle="--", alpha=0.35)
for bar in bars1:
    w = bar.get_width()
    ax1.text(w + 0.003 if w >= 0 else w - 0.003, bar.get_y() + bar.get_height()/2, f"{w:+.4f}", va="center", ha="left" if w >= 0 else "right", fontsize=8, fontweight="bold")

# Native MDI Plot
top_nat = df_native_agg.head(10).iloc[::-1]
bars2 = ax2.barh(y_pos, top_nat["native_importance"], color="#0D9488", alpha=0.85)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(top_nat["feature"], fontsize=10)
ax2.set_xlabel("Mean Decrease in Impurity (Gini Importance)", fontsize=10, fontweight="bold")
ax2.set_title("Native Decision Tree MDI (Training Splits)", fontsize=11, fontweight="bold")
ax2.grid(axis="x", linestyle="--", alpha=0.35)
for bar in bars2:
    w = bar.get_width()
    ax2.text(w + 0.002, bar.get_y() + bar.get_height()/2, f"{w:.4f}", va="center", ha="left", fontsize=8, fontweight="bold")

plt.tight_layout()
plt.show()

C:\Users\ABHINAV DASH\AppData\Local\Temp\ipykernel_13796\1481464543.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
### 8. Explaining Individual Predictions: Decision Paths and Cohort Baselines

To explain individual predictions faithfully without making unsupported causal claims, we trace the exact decision sequence through the fitted tree and compare actionable attributes against reference cohort averages.

In [8]:
print("=" * 70)
print("LOCAL PREDICTION EXPLANATIONS: REPRESENTATIVE TEST SAMPLES")
print("=" * 70)

for tier in ["Low", "Medium", "High"]:
    indices = y_test[y_test == tier].index
    if len(indices) == 0:
        continue
    idx = indices[0]
    rec = X_test.loc[idx]
    exp = explain_individual_prediction(pipeline, rec, reference_df=X_test)

    print(f"\n[Case Study: {tier} Tier Student (Index {idx})]")
    print(f"Actual Tier: {tier} | Predicted Tier: {exp['predicted_category']}")
    print(f"Leaf Probabilities: {exp['predicted_probabilities']}")
    print(f"Traversed Tree Depth: {exp['traversed_depth']} split nodes")
    print("Evaluated Decision Path:")
    for i, rule in enumerate(exp["decision_rules"][:6], 1):
        print(f"  {i}. Node {rule['node_id']}: {rule['rule_description']}")
    print("Cohort Comparison:")
    for feat in ["failures", "absences", "studytime", "Dalc"]:
        stat = exp["cohort_comparison"][feat]
        print(f"  - {feat}: Student = {stat['student_value']} | Cohort Mean = {stat.get('cohort_mean', '—')}")
    break  # Display one in notebook for brevity

LOCAL PREDICTION EXPLANATIONS: REPRESENTATIVE TEST SAMPLES

[Case Study: Low Tier Student (Index 1)]
Actual Tier: Low | Predicted Tier: Low
Leaf Probabilities: {'High': 0.0, 'Low': 1.0, 'Medium': 0.0}
Traversed Tree Depth: 6 split nodes
Evaluated Decision Path:
  1. Node 0: failures = 0.0 (<= 0.50)
  2. Node 1: age = 17.0 (> 16.50)
  3. Node 57: studytime = 1.0 (<= 1.50)
  4. Node 58: Feature 'Mjob_health' was 0.0 (<= 0.50)
  5. Node 59: famrel = 4.0 (> 1.50)
  6. Node 61: Dalc = 5.0 (> 4.50)
Cohort Comparison:
  - failures: Student = 0 | Cohort Mean = 0.2
  - absences: Student = 8 | Cohort Mean = 3.86
  - studytime: Student = 1 | Cohort Mean = 1.84
  - Dalc: Student = 5 | Cohort Mean = 1.64


In [9]:
from src.explainability import save_day24_outputs
paths = save_day24_outputs(n_repeats=15, random_state=42)
print({k: str(v.relative_to(root)) for k,v in paths.items()})

[2026-11-02 13:42:06,024] INFO in model_explainability: Computing permutation importance with 15 repeats (scoring='f1_macro', seed=42)...


[2026-11-02 13:42:09,892] INFO in model_explainability: Saved all Day 24 explainability reports to: C:\Users\ABHINAV DASH\Documents\Codex\2026-11-02\and\work\IYB_INTERNSHIP\reports\day_24


{'permutation_csv': 'reports\\day_24\\permutation_importance.csv', 'permutation_png': 'reports\\day_24\\permutation_importance.png', 'native_csv': 'reports\\day_24\\native_feature_importance.csv', 'native_png': 'reports\\day_24\\native_feature_importance.png', 'individual_json': 'reports\\day_24\\sample_individual_explanations.json'}


## Findings, interpretation and limitations

Use the executed tables above and reports/day_24/permutation_importance.csv for current feature rankings. Earlier Drive prose contained numerical claims that were not independently supported by its saved execution state; this repaired version relies on recalculated outputs.

The 130-row test set has already been examined in earlier project days, so it is a historical evaluation set, not a new independent validation sample. Correlated features share importance. Model confidence is not a calibrated educational probability. Sensitive demographic and health attributes require care; this portfolio demonstration must not determine student access, grading or eligibility.

## Development log and conclusion

Recovered the original Drive notebook on 2 November 2026, restored its dependencies, and reran its code against the verified Day 17 artifact. No hosted deployment has been independently verified.

## Next-day plan

Validate explanation reproducibility and quantify ranking variability on Day 25.
